# Homework 3
-   **Name:**  Victor Hugo Gomez Soto 
-  **e-mail:** -- victor.gomez2701@alumnos.udg.mx --


# MODULES

In [116]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.spatial import distance
from scipy.stats import wrapcauchy, levy_stable
import math


# Functions

In [117]:
# Nota: Esta clase la importaremos junto con el segundo bloque de modulos
################# http://www.pygame.org/wiki/2DVectorClass ##################
class Vec2d(object):
    """2d vector class, supports vector and scalar operators,
       and also provides a bunch of high level functions
       """
    __slots__ = ['x', 'y']

    def __init__(self, x_or_pair, y = None):
        if y == None:            
            self.x = x_or_pair[0]
            self.y = x_or_pair[1]
        else:
            self.x = x_or_pair
            self.y = y
            
    # Addition
    def __add__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x + other.x, self.y + other.y)
        elif hasattr(other, "__getitem__"):
            return Vec2d(self.x + other[0], self.y + other[1])
        else:
            return Vec2d(self.x + other, self.y + other)

    # Subtraction
    def __sub__(self, other):
        if isinstance(other, Vec2d):
            return Vec2d(self.x - other.x, self.y - other.y)
        elif (hasattr(other, "__getitem__")):
            return Vec2d(self.x - other[0], self.y - other[1])
        else:
            return Vec2d(self.x - other, self.y - other)
    
    # Vector length
    def get_length(self):
        return math.sqrt(self.x**2 + self.y**2)
    
    # rotate vector
    def rotated(self, angle):        
        cos = math.cos(angle)
        sin = math.sin(angle)
        x = self.x*cos - self.y*sin
        y = self.x*sin + self.y*cos
        return Vec2d(x, y)
     # Método para convertir el vector en una tupla
    def to_tuple(self):
        return (self.x, self.y)

In [118]:
#####################################################################################
# Brownian motion trajectoy
#####################################################################################
def bm_2d(n_steps=1000, speed=5, s_pos=[0,0]):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle = np.random.uniform(low=-np.pi, high=np.pi)
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.x_pos[i]+velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df
#####################################################################################
# Correlated Random Walk 
#####################################################################################
def rw_2d(n_steps=1000, speed=5, s_pos=[0,0],c =0.5):
    """
    Arguments:
        n_steps:
        speed:
        s_pos:
    Returns:
        BM_2d_df:
    """
    # Init velocity vector
    velocity =Vec2d(speed,0)
    
    # Init DF
    BM_2d_df = pd.DataFrame(columns=['x_pos','y_pos'])    
    # Add initial position
    temp_df = pd.DataFrame([{'x_pos':s_pos[0], 'y_pos':s_pos[1]}])    
    BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
    
    # Generate the trajectory
    for i in range(n_steps-1):        
        turn_angle =   wrapcauchy.rvs(c)   
        velocity = velocity.rotated(turn_angle)
    
        temp_df = pd.DataFrame([{'x_pos':BM_2d_df.iloc[i]['x_pos'] + velocity.x, 'y_pos':BM_2d_df.y_pos[i]+velocity.y}])    
        BM_2d_df = pd.concat([BM_2d_df, temp_df], ignore_index=True)
        
    return BM_2d_df

#####################################################################################
# Correlated Random Walk 
#####################################################################################
def levy_flight(n_steps=1000, alpha=1.5, scale=1.0, c = 0.5):
    pos = Vec2d(0, 0)
    trajectory = [pos.to_tuple()]
    angle = 0  # Ángulo inicial en radianes
    for i in range(n_steps):
        step_size = np.abs(levy_stable.rvs(alpha, 0, scale=scale))  # Tamaño del paso con Lévy
        delta_angle = wrapcauchy.rvs(c)  # Generar un ángulo con distribución de Cauchy
        angle += delta_angle
        step = Vec2d(step_size, 0).rotated(angle)
        pos += step
        trajectory.append(pos.to_tuple())

    print("Primeros 5 puntos de la trayectoria:", trajectory[:5])  # Verifica si hay datos

    x, y = zip(*trajectory)
    z = np.linspace(0, 1, len(x))  # Crear un eje Z para la visualización 3D
    
    print("Cantidad de puntos generados:", len(x))  # Debe ser n_steps + 1

    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode='lines', name='Lévy Flight'))
    fig.show()


# Activity 1: Path length - BM1 vs BM2 vs CRW

In [119]:
###############
# Path Length #
###############
def path_length(trajectory):
    """
    Calculate the total path length of a given trajectory.
    
    Parameters:
    trajectory: A pandas DataFrame containing the trajectory
    
    Returns:
    path_length: A numpy array containing the cumulative sum of the distances
    """
    # Get the Euclidean Distance
    distances = np.array([distance.euclidean(trajectory.iloc[i-1], trajectory.iloc[i]) for i in range(1, trajectory.shape[0])])
    # Get the Cumulative Sum of the stephs
    return np.cumsum(distances)



n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_5": rw_2d(n_steps, speed=5),
    "CRW_6": rw_2d(n_steps, speed=6),
    "Levy_1": levy_flight(n_steps, alpha=1),
    "Levy_07": levy_flight(n_steps, alpha=0.7)
}

# Calcular longitudes de los caminos
# path_lengths = {key: path_length(df) for key, df in walks.items()}
path_lengths = {key: path_length(df) for key, df in walks.items() if df is not None}

# Definir el ancho de línea para cada caso
line_widths = {"BM_3": 2, "BM_6": 8, "CRW_5": 2, "CRW_6": 2, "Levy_1": 2, "Levy_07": 2}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: La caminata '{key}' es None y no se graficará.")
        continue  # Saltar esta iteración si df es None

    fig.add_trace(go.Scatter(
        x=df.index,
        y=path_lengths.get(key, []),  # Evitar error si path_lengths[key] no existe
        marker=dict(size=2),
        line=dict(width=line_widths[key]),
        mode='lines',
        name=f'Path length {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text='Path length - (BM1 vs BM2 vs CRW)',
    autosize=False,
    width=900,
    height=500
)

# Mostrar la gráfica
fig.show()


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(2.4462676768135885), np.float64(-0.5469695966786525)), (np.float64(3.6700259362149508), np.float64(-1.2956795450858483)), (np.float64(10.62480240253672), np.float64(19.925326280583462)), (np.float64(10.893144869330138), np.float64(20.41976406619563))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(1.4375262140420126), np.float64(-0.25839380487500924)), (np.float64(2.3981849946698928), np.float64(-0.8171796020866979)), (np.float64(2.5568952174004984), np.float64(-1.0299135572447151)), (np.float64(10.534889699238184), np.float64(-8.195141275640628))]
Cantidad de puntos generados: 1001


Advertencia: La caminata 'Levy_1' es None y no se graficará.
Advertencia: La caminata 'Levy_07' es None y no se graficará.


# Activity 2: Lévy Distribution - N Different Curves

In [120]:
#############################
# Mean Squared Displacement #
#############################
def msd(trajectory):
    """
    Compute the Mean Squared Displacement (MSD) for a given trajectory.
    
    Parameters:
    trajectory: A numpy array containing the trajectory
    
    Returns:
    msd: The MSD as a function of time.
    """
    N = len(trajectory)    
    msd = np.zeros(N-1)
    
    for i in range(1,N):
        displacements = trajectory[i:] - trajectory[:N - i]
        squared_displacements = np.sum(displacements**2, axis=1) # Square each coordinate of the displacement and add them together
        msd[i-1] = np.mean(squared_displacements) # Save the msd
    
    return msd

n_steps = 1000

# Definir las configuraciones de cada tipo de caminata
walks = {
    "BM_3": bm_2d(n_steps, speed=3),
    "BM_6": bm_2d(n_steps, speed=6),
    "CRW_6_c0.6": rw_2d(n_steps, speed=6, c=0.6),
    "CRW_6_c0.9": rw_2d(n_steps, speed=6, c=0.9),
    "Levy_6_alpha1": levy_flight(n_steps, alpha=1, c=0.5),
    "Levy_6_alpha0.7": levy_flight(n_steps, alpha=0.7, c=0.5)
}
# Verificar si alguna caminata devolvió None
for key, df in walks.items():
    if df is None:
        print(f"Error: La función para '{key}' devolvió None.")

# Obtener las trayectorias (solo columnas x_pos y y_pos)
trajectories = {
    key: df[['x_pos', 'y_pos']].values for key, df in walks.items() if isinstance(df, pd.DataFrame) and not df.empty
}

# Calcular MSD para cada trayectoria
msd_values = {key: msd(traj) for key, traj in trajectories.items()}

# Crear la figura
fig = go.Figure()

# Agregar trazas en un loop
for key, df in walks.items():
    if df is None:
        print(f"Advertencia: {key} es None y no se graficará.")
        continue  # Saltar esta iteración si df es None
    
    fig.add_trace(go.Scatter(
        x=df.index,
        y=msd_values.get(key, []),  # Evitar error si key no está en msd_values
        marker=dict(size=2),
        line=dict(width=2),
        mode='lines',
        name=f'MSD {key.replace("_", " ")}',
        showlegend=True
    ))

# Configuración del layout
fig.update_layout(
    title_text="Mean Squared Displacement - (BM vs CRW)",
    autosize=False,
    width=900,
    height=500,
    xaxis=dict(title="Time Step"),
    yaxis=dict(title="Mean Squared Displacement")
)

# Mostrar la gráfica
fig.show()

Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(1.5356007970117147), np.float64(0.05067464957897982)), (np.float64(2.6568665029886276), np.float64(0.2641186868031826)), (np.float64(-0.25721274053756416), np.float64(3.9582405522285105)), (np.float64(-0.4748355769684188), np.float64(4.191647016893748))]
Cantidad de puntos generados: 1001


Primeros 5 puntos de la trayectoria: [(0, 0), (np.float64(0.3281601523941594), np.float64(0.4926040116481716)), (np.float64(0.3620734581453511), np.float64(1.7184229016234867)), (np.float64(-0.4247962731807168), np.float64(4.724043156216564)), (np.float64(-1.1949868981963783), np.float64(4.8932680486414))]
Cantidad de puntos generados: 1001


Error: La función para 'Levy_6_alpha1' devolvió None.
Error: La función para 'Levy_6_alpha0.7' devolvió None.
Advertencia: Levy_6_alpha1 es None y no se graficará.
Advertencia: Levy_6_alpha0.7 es None y no se graficará.


# Activity 3: Histograms + Curves


# Activity 4:  Step-length Distribution